# bordeus — pipeline di ingestion RAG (LangChain)

Walkthrough interattivo della pipeline, riscritta attorno a LangChain per ogni passaggio.

Gli embedding sono raggruppati per **area Sub-ATO**, non per comune: un gestore può servire più aree con contenuti diversi (es. Quendoz, Valle d'Aosta, gestisce sia il Sub-ATO C sia il D con pagine guida separate) — l'area, non il gestore, è la chiave giusta. Un'area può avere più fonti (`--url`, ripetibile: tipico una pagina specifica + un contenuto condiviso da tutte le aree dello stesso gestore, come un vocabolario) e più comuni.

0. **Area + comuni coperti** — INSERT/UPSERT in tabella `sub_ato` e `comuni` (un'area può servire più comuni, un comune eredita l'area a cui appartiene)
1. **Fetch** — una o più pagine sorgente + PDF/Markdown linkati, scaricati una volta per URL e salvati in `knowledge/<area_id>/<categoria>/`
2. **Load** — `BSHTMLLoader` (HTML) + `PDFPlumberLoader` (PDF) + `TextLoader` (Markdown nativo)
3. **Split** — `MarkdownTextSplitter` per i Markdown nativi, `RecursiveCharacterTextSplitter` (separatori limitati a riga/paragrafo) per HTML/PDF
4. **Embed + scrittura** — `langchain-postgres` (`PGVector`), un'unica collection per l'area (`collection_name = area_id`)
5. **Visualizzazione** — t-SNE 2D/3D degli embedding scritti
6. **Retriever** — `vectorstore.similarity_search(domanda, filter=...)`, con un filtro sul comune specifico (vedi sotto)

Il bot Telegram (`../bot/`, anch'esso Python) legge dallo stesso vector store scritto qui — un solo schema, un solo linguaggio.

Per l'uso non interattivo, `pipeline.run_sub_ato(...)` fa gli step 0-4 in un colpo solo:

```bash
uv run bordeus-ingest \
    --sub-ato=sub-ato-e:"Sub-ATO E — Mont-Rose e Walser" \
    --gestore="TeknoService Italia" \
    --url=https://www.teknoserviceitalia.com/rifiuti \
    --comune=donnas:Donnas --comune=bard:Bard
```

In [ ]:
import logging
import os
import sys

sys.path.insert(0, "../../common/src")
sys.path.insert(0, "../src")

from dotenv import load_dotenv

load_dotenv("../.env")
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s"
)

from bordeus_common import db, embed
from bordeus_common.vectorstore import add_chunks, get_vectorstore
from bordeus_ingest import knowledge as knowledge_mod
from bordeus_ingest import viz
from bordeus_ingest.chunk import split_documents
from bordeus_ingest.loaders import load_knowledge
from bordeus_ingest.pipeline import ComuneInput, fetch_to_knowledge


## Setup: area Sub-ATO, fonti e comuni coperti

Un'area Sub-ATO (`area_id`) può avere più fonti (`source_urls`, tipico: una pagina specifica dell'area + un contenuto condiviso da tutte le aree dello stesso gestore, come un vocabolario) e più comuni (`comuni`). Tutti i comuni della stessa area condividono la stessa collection nel vector store: i dati di aree diverse non si mescolano mai, anche se condividono lo stesso gestore.

In [ ]:
database_url = os.environ["DATABASE_URL"]

area_id = "sub-ato-d"                                          # cambia con lo id reale
area_nome = "Sub-ATO E — Mont-Rose e Walser"                  # cambia con il nome reale
gestore = "TeknoService Italia"                                                # cambia con il nome reale
source_urls = [
    "https://www.quendoz.it/category/subato-d/",                  # pagina specifica dell'area
    "https://www.quendoz.it/vocabolario/",                        # contenuto condiviso da tutte le aree di questo gestore
]

comuni = [
    ComuneInput(id="donnas", nome="Donnas"),
    ComuneInput(id="verres", nome="Verrès"),
]

# Contenuto SPECIFICO di un comune (non condiviso dall'area): tipico un
# calendario di raccolta, che può variare da un comune all'altro anche
# sotto lo stesso gestore. Opzionale — lascia vuoto {} se non serve.
comune_urls = {
    "montjovet": ["https://www.quendoz.it/calendario-montjovet.pdf"],
    "verres": ["https://www.quendoz.it/calendario-verres.pdf"],
}

## Step 0 — INSERT/UPSERT di area e comuni in Postgres

Né l'area né i comuni devono esistere già: l'ingestion li crea (o aggiorna nome/gestore se già presenti) prima di scrivere qualunque chunk. Stesso comportamento `ON CONFLICT (id) DO UPDATE` usato dal bot quando legge le stesse tabelle.

In [ ]:
conn = db.connect(database_url)

db.upsert_sub_ato(conn, area_id, area_nome, gestore)
print(f"area registrata: {area_id} ({area_nome!r}, gestore={gestore!r})")

for c in comuni:
    db.upsert_comune(conn, c.id, c.nome, area_id)
    print(f"comune registrato: {c.id} ({c.nome!r}) -> {area_id}")

## Step 1 — Fetch (una volta per URL) → `knowledge/<area_id>/`

`fetch_to_knowledge` scarica ciascuna pagina sorgente e tutti i PDF/Markdown linkati (un giro di rete per URL), li classifica per categoria (calendario/guide/moduli/servizi/altro — euristica in `classify.py`, anche dal contenuto per i PDF, non solo dal nome file) e li salva in un'unica cartella `knowledge/<area_id>/<categoria>/`, condivisa da tutti i comuni dell'area.

In [ ]:
total = fetch_to_knowledge(area_id, source_urls, comune_urls=comune_urls)
print(f"File salvati: {total}\n")

base_dir = knowledge_mod.area_dir(area_id)
print(f"knowledge/{area_id}/")
for p in sorted(base_dir.rglob("*")):
    if p.is_file() and p.name != "manifest.json":
        print("  ", p.relative_to(base_dir))

## Step 2-4

Il resto della pipeline (load → split → embed → scrittura) opera sull'intera area: un'unica collection nel vector store, condivisa da tutti i comuni che vi appartengono. `pipeline.run_sub_ato(...)` fa lo stesso in un colpo solo (vedi la cella dopo il retriever).

In [ ]:
documents = load_knowledge(area_id)
print(f"Step 2 — {len(documents)} Document caricati\n")
for d in documents[:5]:
    print(f"[{d.metadata['kind']:9} | {d.metadata['tipo']:10}] {d.page_content[:80]!r}")

In [ ]:
chunks = split_documents(documents)
print(f"Step 3 — {len(chunks)} chunk generati\n")
if chunks:
    print("Primo chunk:\n", chunks[0].page_content[:300])
    print("\nMetadata:", chunks[0].metadata)

In [ ]:
embeddings = embed.get_embeddings()
vectorstore = get_vectorstore(database_url, area_id, embeddings)

ids = add_chunks(vectorstore, chunks)
print(f"Step 4 — {len(ids)} chunk scritti su Postgres (collection={area_id!r})")

## Step 5 — Visualizzazione degli embedding (t-SNE, 2D e 3D)

Riduciamo gli embedding a 2 o 3 dimensioni per ispezionarli visivamente — utile per notare anomalie evidenti (es. un intero documento isolato dagli altri, segno di un'estrazione andata male).

In [ ]:
records = viz.fetch_embeddings_for_collection(database_url, area_id)
print(f"{len(records)} chunk trovati per la collection {area_id!r}")

embeddings_list = [r["embedding"] for r in records]
labels_tipo = [r["metadata"].get("tipo", "n/d") for r in records]

coords_2d = viz.reduce(embeddings_list, n_components=2)
fig_2d = viz.plot(coords_2d, labels_tipo, title=f"Chunk embedding — {area_id} (2D, colorato per categoria)")

In [ ]:
coords_3d = viz.reduce(embeddings_list, n_components=3)
fig_3d = viz.plot(coords_3d, labels_tipo, title=f"Chunk embedding — {area_id} (3D, colorato per categoria)")

## Step 6 — Test del retriever, con filtro per comune

Non tutto il contenuto di un'area è condiviso da tutti i comuni che la compongono (vedi `--comune-url` più sopra nel README): un filtro sui metadata del chunk (`comune_id`) restituisce il contenuto condiviso dell'area **più** quello specifico di UN comune, mai quello di un comune vicino. `bot/rag.py` (`comune_filter`) fa esattamente questo in produzione — qui lo replichiamo a mano per ispezionare il comportamento.

In [ ]:
def comune_filter(comune_id: str) -> dict:
    return {"$or": [{"comune_id": ""}, {"comune_id": comune_id}]}

domanda = "Come devo smaltire una bottiglia di plastica?"   # prova con una domanda pertinente alla tua area

# Sostituisci con uno id di comune reale della tua area (o "" per
# vedere solo il contenuto condiviso, senza nulla di specifico)
comune_id_prova = comuni[0].id

risultati = vectorstore.similarity_search(domanda, k=4, filter=comune_filter(comune_id_prova))

print(f"Domanda: {domanda}")
print(f"Filtrato per comune: {comune_id_prova!r}\n")
for r in risultati:
    comune_tag = r.metadata.get("comune_id") or "area condivisa"
    print(f"[{r.metadata.get('tipo')} | {comune_tag}] {r.page_content[:200]!r}")
    print(f"   fonte: {r.metadata.get('source_url')}\n")

## Tutto insieme: `pipeline.run_sub_ato`

Equivalente non interattivo degli step 0-4 sopra, per l'intera area in un colpo solo — quello che gira dietro `uv run bordeus-ingest`. Restituisce un solo `PGVector` (un'unica collection per l'intera area, non più un dict per comune).

In [ ]:
from bordeus_ingest.pipeline import run_sub_ato

vectorstore = run_sub_ato(
    area_id, area_nome, gestore, comuni, source_urls, database_url, embeddings,
    comune_urls=comune_urls,
)
print(f"Collection creata/aggiornata: {area_id!r}")